# 6. Off-Policy Monte Carlo Control
### Learns a Target Policy Using Another Behavior Policy

**Off-Policy Monte Carlo Control** evaluates and optimizes a deterministic **target policy $\pi$** while generating experience from an exploratory **behavior policy $b$**.

#### Key Concepts:
1. **Target Policy $\pi$**: Greedy with respect to current $Q(s, a)$.
2. **Behavior Policy $b$**: Exploratory (e.g. uniform random or $\epsilon$-greedy) to ensure all actions are sampled.
3. **Importance Sampling Ratio**:
   $$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k \mid S_k)}{b(A_k \mid S_k)}$$
4. **Early Termination of Backward Pass**:
   Since $\pi$ is deterministic, if the behavior action $A_t \neq \pi(S_t)$, then $\pi(A_t \mid S_t) = 0$, causing all subsequent importance sampling weights to become 0!


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Pure Python Blackjack Environment (Sutton & Barto Ch. 5)
# State: (player_sum: 12..21, dealer_showing_card: 1..10, usable_ace: bool)
# Actions: 0 = Stick, 1 = Hit

class BlackjackEnv:
    def __init__(self):
        self.actions = [0, 1]  # 0: Stick, 1: Hit
        
    def _draw_card(self):
        card = np.random.randint(1, 14)
        return min(card, 10)  # Face cards (J, Q, K) have value 10; Ace is 1
        
    def _draw_hand(self):
        return [self._draw_card(), self._draw_card()]
        
    def _get_hand_value(self, hand):
        val = sum(hand)
        has_ace = 1 in hand
        if has_ace and val + 10 <= 21:
            return val + 10, True  # Usable Ace
        return val, False
        
    def reset(self, start_state=None, start_action=None):
        if start_state is not None:
            player_sum, dealer_card, usable_ace = start_state
            self.player_hand = [1, player_sum - 11] if usable_ace else [player_sum // 2, player_sum - player_sum // 2]
            self.dealer_hand = [dealer_card, self._draw_card()]
        else:
            self.player_hand = self._draw_hand()
            while self._get_hand_value(self.player_hand)[0] < 12:
                self.player_hand.append(self._draw_card())
            self.dealer_hand = self._draw_hand()
            
        p_val, usable_ace = self._get_hand_value(self.player_hand)
        return (p_val, self.dealer_hand[0], usable_ace)
        
    def step(self, action):
        if action == 1:  # Hit
            self.player_hand.append(self._draw_card())
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            if p_val > 21:
                return (p_val, self.dealer_hand[0], usable_ace), -1.0, True  # Bust
            return (p_val, self.dealer_hand[0], usable_ace), 0.0, False
        else:  # Stick: Dealer plays
            while True:
                d_val, _ = self._get_hand_value(self.dealer_hand)
                if d_val >= 17:
                    break
                self.dealer_hand.append(self._draw_card())
                
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            d_val, _ = self._get_hand_value(self.dealer_hand)
            
            if d_val > 21 or p_val > d_val:
                reward = 1.0  # Player wins
            elif p_val == d_val:
                reward = 0.0  # Draw
            else:
                reward = -1.0 # Dealer wins
                
            return (p_val, self.dealer_hand[0], usable_ace), reward, True

env = BlackjackEnv()
print("Blackjack environment initialized.")


In [ ]:
# Step 3: Implement Off-Policy MC Control with Weighted Importance Sampling
def off_policy_mc_control(env, num_episodes=100000, gamma=1.0):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    C = defaultdict(lambda: np.zeros(len(env.actions))) # Cumulative sum of weights
    pi = defaultdict(int) # Target policy is greedy
    
    for ep in range(1, num_episodes + 1):
        # Generate episode using soft behavior policy b (equal probability 0.5)
        episode = []
        s = env.reset()
        while True:
            # Behavior policy b: uniform random over actions
            a = np.random.choice(env.actions)
            next_s, r, done = env.step(a)
            episode.append((s, a, r))
            s = next_s
            if done:
                break
                
        G = 0.0
        W = 1.0 # Importance sampling weight
        
        # Traverse episode in reverse
        for t in reversed(range(len(episode))):
            s_t, a_t, r_t = episode[t]
            G = gamma * G + r_t
            
            C[s_t][a_t] += W
            Q[s_t][a_t] += (W / C[s_t][a_t]) * (G - Q[s_t][a_t])
            
            # Target policy: greedy action
            pi[s_t] = np.argmax(Q[s_t])
            
            # If behavior action was not the target policy action, weight becomes 0!
            if a_t != pi[s_t]:
                break
                
            # Update importance weight: W = W * (1.0 / b(a|s)) where b(a|s) = 0.5
            W *= 2.0
            
    return Q, pi

Q_off, pi_target = off_policy_mc_control(env, num_episodes=80000)
print("Off-Policy MC Control training complete.")


In [ ]:
# Step 4: Evaluate the Learned Deterministic Target Policy
test_episodes = 10000
wins, draws, losses = 0, 0, 0

for _ in range(test_episodes):
    s = env.reset()
    while True:
        a = pi_target[s]
        s, r, done = env.step(a)
        if done:
            if r > 0: wins += 1
            elif r == 0: draws += 1
            else: losses += 1
            break

print(f"Target Policy Evaluation ({test_episodes} games):")
print(f"Win Rate:  {wins / test_episodes * 100:.2f}%")
print(f"Draw Rate: {draws / test_episodes * 100:.2f}%")
print(f"Loss Rate: {losses / test_episodes * 100:.2f}%")


### Summary
- Separates exploration (behavior policy $b$) from optimal control (target policy $\pi$).
- Can learn from existing logged data or human demonstrations.
